# Argus — YOLO nano baseline (Ultralytics)

The off-the-shelf reference for our from-scratch detector in `12_tiny_grid_detector.ipynb`: a
**pretrained YOLO nano** (COCO weights), fine-tuned on the same face-crop dataset, splits and
classes, and scored by the same evaluation code. If our tiny CNN gets close to this at a
fraction of the size and latency, that's the result.

Two models, trained as two separate steps:
- **Step 1, `yolo11n`**: the current Ultralytics nano, about 2.6 M parameters. Primary.
- **Step 2, `yolov5nu`**: the YOLOv5-nano lineage (anchor-free "u" head), about 2.6 M parameters.

**How the notebook is laid out:**
1. Setup, loading, and the shared evaluation / latency cells (shared verbatim with 12 / 14).
2. **Checking the labelled dataset**: class and box counts, where the head labels come from
   (human head-pose label vs. landmark pitch), why images were rejected, and a grid of labelled
   examples per class.
3. **Step 1 (`yolo11n`)** and **Step 2 (`yolov5nu`)**. Each one trains, then gets its own test
   predictions, per-class precision / recall, a confusion matrix, and predicted-vs-labelled
   images, including the images it gets most wrong.
4. **Step 1 vs. Step 2**: one table, and the same test images through both models side by side.

Each step can be run on its own (e.g. only Step 2 after a disconnect), as long as sections 1–2
and the "Plots" cell have been run first.

**Input.** The same `IMG = 160` face crops. Ultralytics reads the 1-channel JPEGs as 3-channel
BGR (gray replicated), so the network sees gray. HSV augmentation is turned off, since there is
no colour to jitter.

**Licence note.** Ultralytics is **AGPL-3.0**. That's fine for this academic project, but it
matters before any commercial deployment.

**Reads:** `dataset/parts_det.zip`.

**Writes:**
- `models/parts_det/<model>_*` (best `.pt` + int8 `.tflite`);
- `<name>_metrics.json` for notebook 14's comparison table.
- `figures/` next to the models: the dataset-check and per-step plots as PNG.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9"], check=False)

## PyTorch gets the GPU; TensorFlow stays on the CPU

This notebook trains with **Ultralytics, i.e. PyTorch**, but the shared cells also `import
tensorflow`, only to time the int8 `.tflite` on the CPU. On Colab, letting both frameworks load
their CUDA/cuDNN stacks in one kernel crashed it: a silent "kernel restarted" about 20 s after
TensorFlow loaded, right when YOLO training started.

So PyTorch initialises the GPU **first**, and TensorFlow is then imported with **every GPU
hidden from it**. The shared latency cell's later `import tensorflow` reuses this CPU-only
instance. Run this cell before any other.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import torch
if torch.cuda.is_available():
    torch.zeros(1, device="cuda")              # create PyTorch's CUDA context before TF loads
    print("PyTorch GPU:", torch.cuda.get_device_name(0))
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")      # TF: CPU only (tflite timing doesn't need a GPU)
print("TensorFlow devices:", tf.config.get_visible_devices())

## Setup

**Getting the dataset in (Colab).** Upload `parts_det.zip`, built by
`src/dataset/scripts/build_detection_dataset.py`, **straight into the session**. Either drag it
into the Files panel (left sidebar → 📁 → upload, so it lands at `/content/parts_det.zip`), or
run the Loading cell and use the upload button it shows if the zip isn't found. Drive isn't
needed for the dataset.

**Keeping the results.** A Colab session's disk is wiped when it ends.
- With `USE_DRIVE = True` (the default), Drive is mounted only to save trained models and
  metrics to `MyDrive/Argus/models/parts_det/`. They survive the session, and notebook 14's
  comparison table can read all three notebooks' results.
- With `USE_DRIVE = False`, everything is saved under `/content/models/parts_det/` and the last
  cell zips it for download.

**Locally:** set `ARGUS_DET_ZIP`, and optionally `ARGUS_PROJECT_FOLDER` / `ARGUS_WORK_DIR`.
`ARGUS_SMOKE=1` turns every training run into a few-step smoke test on a small subset. It's used
to run the notebook end to end on a CPU before spending GPU time.

**This cell is shared verbatim by notebooks 12 / 13 / 14** so their numbers are comparable;
change it in all three or none.

In [ ]:
import os
import sys
import json
import time
import shutil
import zipfile
import datetime

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("ARGUS_SMOKE", "0") == "1"
USE_DRIVE = True          # Colab only: save models/metrics to Drive so they outlive the session

if IN_COLAB:
    WORK_DIR = "/content"
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        project_folder = "/content/drive/MyDrive/Argus"
    else:
        project_folder = "/content"
else:
    project_folder = os.environ.get("ARGUS_PROJECT_FOLDER", os.path.expanduser("~/argus-colab-mirror"))
    WORK_DIR = os.environ.get("ARGUS_WORK_DIR", "/tmp/argus_work")

models_folder = f"{project_folder}/models/parts_det"
# Where the dataset zip may be, in order: explicit env var, uploaded into the session, on Drive.
DET_ZIP_CANDIDATES = [p for p in (os.environ.get("ARGUS_DET_ZIP"), f"{WORK_DIR}/parts_det.zip",
                                  f"{project_folder}/dataset/parts_det.zip") if p]
os.makedirs(models_folder, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# Dataset contract (src/dataset/argus_dataset/config.py: DET_CLASSES, DET_IMG_SIZE = 192 stored).
CLASSES = ["open_eye", "closed_eye", "yawn", "head_straight", "head_down"]
NUM_CLASSES = len(CLASSES)
EYE_CLASSES, YAWN_CLASS, HEAD_CLASSES = (0, 1), 2, (3, 4)

IMG = 160                 # model input side (stored crops are 192 px, resized here)
VARIANT = "mix"           # "gray" | "ir" | "mix" (train: random gray/IR blend per image)
EVAL_VARIANT = "ir"       # what val/test images look like: the cabin camera is a Pi NoIR
INCLUDE_CLOSEUPS = True   # MRL + Roboflow eye/mouth close-ups (whole image = one box)
INCLUDE_PSEUDO_EYES = True  # images whose skipped eyes were pseudo-labelled from landmarks
CONF_THR = 0.25           # operating point for P/R, eye-state and head accuracy
SEED = 42
np.random.seed(SEED)
print(f"IN_COLAB={IN_COLAB} SMOKE={SMOKE} USE_DRIVE={USE_DRIVE}\nmodels -> {models_folder}")

## Loading the detection dataset

The zip is found in this order:
1. `ARGUS_DET_ZIP`;
2. `/content/parts_det.zip`, i.e. uploaded into the session;
3. `MyDrive/Argus/dataset/parts_det.zip`;
4. otherwise an upload button.

It's unzipped to local session disk once. Training reads the local files, never Drive's FUSE
mount: thousands of small reads off Drive time out, as `07`/`11` found.

The zip is YOLO format:
- `gray/` and `ir/`, each with `{train,valid,test}/{images,labels}`;
- `index.csv`, one row per source image with `status`, `kind`, `source`, `group_id` and `split`.

**Splits are by group:** an original image plus its augmented copies and near-duplicates, or one
MRL subject. They were assigned when the dataset was built. Never re-split by image here.

Each label line is `class cx cy w h`, normalised to 0–1.

**This cell is shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
LOCAL_DIR = f"{WORK_DIR}/parts_det"


def find_dataset_zip():
    for p in DET_ZIP_CANDIDATES:
        if os.path.isfile(p) and zipfile.is_zipfile(p):
            return p
    if IN_COLAB:   # not uploaded yet: show an upload button (Files panel drag-and-drop is faster for 371 MB)
        from google.colab import files
        print("parts_det.zip not found — choose it in the upload button below.")
        for name in files.upload():
            if name.endswith(".zip"):
                dest = f"{WORK_DIR}/parts_det.zip"
                if os.path.abspath(name) != dest:
                    shutil.move(name, dest)
                return dest
    raise FileNotFoundError(f"parts_det.zip not found in {DET_ZIP_CANDIDATES}")


if not os.path.isfile(f"{LOCAL_DIR}/index.csv"):
    DET_ZIP = find_dataset_zip()
    print(f"Extracting {DET_ZIP} -> {LOCAL_DIR}")
    with zipfile.ZipFile(DET_ZIP) as zf:
        zf.extractall(LOCAL_DIR)

index = pd.read_csv(f"{LOCAL_DIR}/index.csv", dtype=str, keep_default_na=False)
ok = index[index.status == "ok"].copy()
if not INCLUDE_CLOSEUPS:
    ok = ok[ok.kind != "closeup"]
if not INCLUDE_PSEUDO_EYES:
    ok = ok[ok.pseudo_eyes.isin(["", "0"])]


def read_labels(label_rel):
    """YOLO label file -> (classes int32 (n,), boxes float32 (n,4) as cx,cy,w,h)."""
    rows = [l.split() for l in open(f"{LOCAL_DIR}/{label_rel}").read().splitlines() if l.strip()]
    if not rows:
        return np.zeros(0, np.int32), np.zeros((0, 4), np.float32)
    a = np.array(rows, np.float32)
    return a[:, 0].astype(np.int32), a[:, 1:5]


splits = {}
for s in ("train", "valid", "test"):
    df = ok[ok.split == s].reset_index(drop=True)
    if SMOKE:   # a few images of every (source, kind) so every code path still runs
        df = (df.groupby(["source", "kind"], group_keys=False)
                .apply(lambda g: g.sample(min(len(g), 24), random_state=SEED)).reset_index(drop=True))
    labels = [read_labels(p) for p in df.label_path]
    df["classes"] = [c for c, _ in labels]
    df["boxes"] = [b for _, b in labels]
    splits[s] = df

summary = pd.DataFrame({
    s: {"images": len(d), **{c: int(sum(int((a == i).sum()) for a in d.classes))
                             for i, c in enumerate(CLASSES)}}
    for s, d in splits.items()})
print(summary)


def image_path(row, variant):
    return f"{LOCAL_DIR}/{row['gray_path'] if variant == 'gray' else row['ir_path']}"


def gts_for(df):
    """Ground truth per image as xyxy boxes, for the shared evaluation."""
    out = {}
    for r in df.itertuples():
        b = r.boxes
        out[r.image_id] = (np.c_[b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                                 b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], r.classes)
    return out

## Checking the labelled dataset

Before spending GPU time, look at what the model will learn from:
- **Boxes per class and split.** `head_down` and `yawn` are the rare classes; if one is nearly
  empty in `valid`/`test`, its AP is noise.
- **Where head labels come from.** `index.csv`'s `head_source` says whether a head box is a
  **human** head-pose label (Roboflow head-pose sources, e.g. `headpose-w4zuc`), a human label on a
  crop made around the human head box because BlazeFace missed the face (`human_box`), or the
  **landmark** pitch rule (`landmarks`). Where a human label and the pitch rule both exist, the
  cross-table shows how often they agree, which is the check on the `HEAD_*_DEG` thresholds.
  Zips built before head-pose sources were added have no such column; the cell skips it.
- **Rejected images and why** (from `index.csv`), so a sudden jump in one reason is visible.
- **Box sizes per class**, relative to the crop side.

Figures are saved under `models/parts_det/figures/` so they survive the session with Drive on.

In [ ]:
import matplotlib.pyplot as plt
import cv2

FIG_DIR = f"{models_folder}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
CLASS_COLORS = plt.cm.tab10(np.arange(NUM_CLASSES))[:, :3]


def savefig(fig, name):
    path = f"{FIG_DIR}/{name}_{VERSION_STR}.png"
    fig.savefig(path, dpi=120, bbox_inches="tight")
    print(f"-> {path}")


# Boxes per class and split
box_counts = summary.drop(index="images")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
box_counts.plot.bar(ax=axes[0], rot=20, title="boxes per class and split")
(box_counts / box_counts.sum()).plot.bar(ax=axes[1], rot=20, title="share of each split's boxes")
plt.tight_layout(); savefig(fig, "dataset_boxes_per_class"); plt.show()

print("Images per source and split:")
print(pd.crosstab(ok.project, ok.split, margins=True).to_string(), "\n")

if "head_source" in ok.columns:
    heads = ok[ok.head_class != ""]
    print("Head boxes by class and label source:")
    print(pd.crosstab(heads.head_class, heads.head_source.replace("", "?"), margins=True).to_string(), "\n")
    human = ok[ok.head_source == "human"]
    if len(human):
        print("Human head label (rows) vs. what the landmark pitch rule says (columns):")
        print(pd.crosstab(human.head_class.replace("", "none (turned/other)"),
                          human.lm_head_class.replace("", "none (ambiguous band)")).to_string(), "\n")
else:
    print("index.csv has no head_source column (zip built before head-pose sources); skipping.\n")

rej = index[index.status != "ok"]
print(f"Rejected source images: {len(rej)} of {len(index)}")
print(rej.groupby(["project", "reason"]).size().unstack(fill_value=0).T.to_string(), "\n")

# Box size (sqrt of area, as a fraction of the crop side) per class, train split
sizes = {c: [] for c in CLASSES}
for cl, bx in zip(splits["train"].classes, splits["train"].boxes):
    for c, b in zip(cl, bx):
        sizes[CLASSES[c]].append(float(np.sqrt(b[2] * b[3])))
fig, ax = plt.subplots(figsize=(8, 3.5))
for i, c in enumerate(CLASSES):
    if sizes[c]:
        ax.hist(sizes[c], bins=40, range=(0, 1), histtype="step", lw=1.5, color=CLASS_COLORS[i], label=c)
ax.set(xlabel="sqrt(box area) / crop side", ylabel="boxes", title="box size per class (train)")
ax.legend(); savefig(fig, "dataset_box_sizes"); plt.show()

### Labelled examples

A few training images per class, as the model sees them (`EVAL_VARIANT`, i.e. IR-simulated
unless changed), with **every** ground-truth box drawn, not just the row's class. Read it for
label problems a table can't show: a skipped eye (it would be learned as background), a head box
on the wrong person, a `head_down` that is really a head turned away. The title says where the
image came from and, for head classes, who labelled the head.

The chosen image ids are written to `labelled_examples_<version>.csv`, so the same examples can
be pulled up again.

In [ ]:
N_EXAMPLES = 6     # images per class


def load_rgb(row, variant=EVAL_VARIANT):
    im = cv2.imread(image_path(row, variant), cv2.IMREAD_GRAYSCALE)
    return cv2.cvtColor(im, cv2.COLOR_GRAY2RGB)


def draw_boxes(ax, img, xyxy, classes, scores=None, title=""):
    """Boxes in normalised xyxy on `img`, coloured by class, with class (and score) text."""
    ax.imshow(img)
    H, W = img.shape[:2]
    for k, (b, c) in enumerate(zip(xyxy, classes)):
        x0, y0, x1, y1 = b[0] * W, b[1] * H, b[2] * W, b[3] * H
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1.4, color=CLASS_COLORS[c]))
        txt = CLASSES[c] if scores is None else f"{CLASSES[c]} {scores[k]:.2f}"
        ax.text(x0, max(y0 - 2, 6), txt, fontsize=6, color="white",
                bbox=dict(facecolor=CLASS_COLORS[c], pad=0.5, lw=0))
    ax.set_title(title, fontsize=7)
    ax.axis("off")


def gt_xyxy(row):
    b = row["boxes"]
    return np.c_[b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2, b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2]


train_df = splits["train"]
fig, axes = plt.subplots(NUM_CLASSES, N_EXAMPLES, figsize=(2.1 * N_EXAMPLES, 2.3 * NUM_CLASSES))
chosen = []
for i, c in enumerate(CLASSES):
    cand = train_df[train_df.classes.apply(lambda a: bool((a == i).any()))]
    pick = cand.sample(min(N_EXAMPLES, len(cand)), random_state=SEED) if len(cand) else cand
    for j in range(N_EXAMPLES):
        ax = axes[i, j]
        if j >= len(pick):
            ax.axis("off")
            if j == 0:
                ax.text(0.5, 0.5, f"no {c} boxes\nin this split", ha="center", va="center", fontsize=9)
            continue
        r = pick.iloc[j]
        who = f" ({r['head_source']})" if i in HEAD_CLASSES and r.get("head_source") else ""
        draw_boxes(ax, load_rgb(r), gt_xyxy(r), r["classes"], title=f"{c}{who}\n{r['project']}")
        chosen.append({"class": c, "image_id": r["image_id"], "project": r["project"],
                       "split": "train", "path": image_path(r, EVAL_VARIANT)})
plt.tight_layout(); savefig(fig, "labelled_examples"); plt.show()
pd.DataFrame(chosen).to_csv(f"{FIG_DIR}/labelled_examples_{VERSION_STR}.csv", index=False)

## Shared evaluation

Each model's output is converted to a common form: per image, `xyxy` boxes (0–1), scores and
class ids, after NMS. It is then scored here, so 12, 13 and 14 report **identically defined**
numbers.

- **mAP@0.5 and mAP@0.5:0.95** use COCO-style 101-point interpolation, per class.
- **P / R per class** are taken at `CONF_THR` and IoU ≥ 0.5.
- **Eye-state accuracy** is what the temporal stage will actually consume. Each ground-truth eye
  is matched to the best-IoU predicted eye box of either eye class (IoU ≥ 0.5, score ≥
  `CONF_THR`). From that we get:
  - `eye_recall`: the share of eyes found;
  - `eye_state_acc`: the open/closed correctness among found eyes;
  - `eye_state_overall`: correct over all eyes.
- **Head accuracy** is defined the same way over `head_straight` / `head_down`.

`report()` also splits these by `kind` (face / closeup) and `source` (roboflow / mrl_eye),
because MRL close-ups are easy and must not hide weak face-crop results.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
def iou_xyxy(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def nms_per_class(boxes, scores, classes, iou_thr=0.5, max_det=30):
    keep = []
    for c in np.unique(classes):
        idx = np.where(classes == c)[0]
        idx = idx[np.argsort(-scores[idx])]
        while len(idx):
            i = idx[0]
            keep.append(i)
            if len(idx) == 1:
                break
            idx = idx[1:][iou_xyxy(boxes[i:i + 1], boxes[idx[1:]])[0] < iou_thr]
    keep = np.array(sorted(keep, key=lambda i: -scores[i])[:max_det], np.int64)
    return boxes[keep], scores[keep], classes[keep]


def _ap(recall, precision):
    r = np.concatenate([[0.0], recall, [1.0]]); p = np.concatenate([[1.0], precision, [0.0]])
    p = np.flip(np.maximum.accumulate(np.flip(p)))
    return float(np.mean(np.interp(np.linspace(0, 1, 101), r, p)))


def average_precision(preds, gts, cls, iou_thrs):
    """AP of one class at each IoU threshold over a set of images."""
    items, n_gt = [], 0
    for iid, (gb, gc) in gts.items():
        g = gb[gc == cls]; n_gt += len(g)
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        m = pc == cls
        items.append((g, pb[m], ps[m]))
    if n_gt == 0:
        return [float("nan")] * len(iou_thrs)
    out = []
    for t in iou_thrs:
        scores, tps = [], []
        for g, pb, ps in items:
            order = np.argsort(-ps); pb, ps = pb[order], ps[order]
            ious = iou_xyxy(pb, g); used = np.zeros(len(g), bool)
            for k in range(len(pb)):
                j = int(np.argmax(ious[k])) if len(g) else -1
                hit = j >= 0 and ious[k, j] >= t and not used[j]
                if hit:
                    used[j] = True
                scores.append(ps[k]); tps.append(hit)
        if not scores:
            out.append(0.0); continue
        order = np.argsort(-np.array(scores)); tp = np.array(tps)[order]
        ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
        out.append(_ap(ctp / n_gt, ctp / np.maximum(ctp + cfp, 1)))
    return out


def state_accuracy(preds, gts, group, conf=CONF_THR, iou_thr=0.5):
    """For GT boxes whose class is in `group`: found-rate and state accuracy (see markdown)."""
    total = found = correct = 0
    for iid, (gb, gc) in gts.items():
        m = np.isin(gc, group)
        if not m.any():
            continue
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        k = np.isin(pc, group) & (ps >= conf)
        ious = iou_xyxy(gb[m], pb[k])
        for g_i, cls in enumerate(gc[m]):
            total += 1
            if ious.shape[1] and ious[g_i].max() >= iou_thr:
                found += 1
                correct += int(pc[k][int(np.argmax(ious[g_i]))] == cls)
    return {"n": total, "recall": found / max(total, 1), "state_acc": correct / max(found, 1),
            "state_overall": correct / max(total, 1)}


def pr_at_conf(preds, gts, cls, conf=CONF_THR, iou_thr=0.5):
    tp = fp = n_gt = 0
    for iid, (gb, gc) in gts.items():
        g = gb[gc == cls]; n_gt += len(g)
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        p = pb[(pc == cls) & (ps >= conf)]
        ious = iou_xyxy(p, g); used = np.zeros(len(g), bool)
        for k in range(len(p)):
            j = int(np.argmax(ious[k])) if len(g) else -1
            if j >= 0 and ious[k, j] >= iou_thr and not used[j]:
                used[j] = True; tp += 1
            else:
                fp += 1
    return tp / max(tp + fp, 1), tp / max(n_gt, 1)


def evaluate(preds, gts):
    thrs = np.arange(0.5, 0.96, 0.05)
    aps = {c: average_precision(preds, gts, i, thrs) for i, c in enumerate(CLASSES)}
    res = {
        "map50": float(np.nanmean([a[0] for a in aps.values()])),
        "map50_95": float(np.nanmean([np.mean(a) for a in aps.values()])),
        "ap50": {c: a[0] for c, a in aps.items()},
        "pr_at_conf": {c: dict(zip(("precision", "recall"), pr_at_conf(preds, gts, i)))
                       for i, c in enumerate(CLASSES)},
        "eye": state_accuracy(preds, gts, EYE_CLASSES),
        "head": state_accuracy(preds, gts, HEAD_CLASSES),
    }
    return res


def report(preds, df, title=""):
    """Overall + per-kind / per-source evaluation of `preds` on the images of `df`."""
    out = {"all": evaluate(preds, gts_for(df))}
    for col in ("kind", "source"):
        for v, sub in df.groupby(col):
            out[f"{col}={v}"] = evaluate(preds, gts_for(sub))
    rows = {k: {"mAP50": r["map50"], "mAP50-95": r["map50_95"],
                **{f"AP50 {c}": r["ap50"][c] for c in CLASSES},
                "eye recall": r["eye"]["recall"], "eye state acc": r["eye"]["state_acc"],
                "head recall": r["head"]["recall"], "head acc": r["head"]["state_acc"]}
            for k, r in out.items()}
    print(f"== {title}")
    print(pd.DataFrame(rows).T.round(3).to_string())
    return out

## Shared latency benchmark and metrics file

The Pi runs **int8 TFLite**. `bench_tflite` times one image through a `.tflite` model with
`num_threads` set to 1, 2 and 4: the Pi 5 has 4 cores, and small models often don't speed up
with more threads, which is the MobileNet-on-Pi question. Colab's CPU is only a *proxy*. The
saved `.tflite` is what gets timed on the real Pi with `benchmark_model`.

`save_metrics` writes `models/parts_det/<name>_metrics.json`. Notebook 14's last cell reads
every such file into one comparison table.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
import tensorflow as tf


def bench_tflite(path, threads=(1, 2, 4), runs=200, warmup=20):
    out = {}
    for n in threads:
        it = tf.lite.Interpreter(model_path=str(path), num_threads=n)
        it.allocate_tensors()
        inp = it.get_input_details()[0]
        x = np.random.rand(*inp["shape"]).astype(np.float32)
        if inp["dtype"] != np.float32:   # integer-input model: feed in its quantised domain
            s, z = inp["quantization"]
            x = np.clip(np.round(x / s + z), np.iinfo(inp["dtype"]).min, np.iinfo(inp["dtype"]).max).astype(inp["dtype"])
        ts = []
        for k in range(warmup + (5 if SMOKE else runs)):
            t0 = time.perf_counter()
            it.set_tensor(inp["index"], x); it.invoke()
            if k >= warmup:
                ts.append((time.perf_counter() - t0) * 1000)
        out[str(n)] = round(float(np.median(ts)), 3)
    print(f"latency (median ms) {os.path.basename(str(path))}: {out}")
    return out


def _jsonable(o):
    if isinstance(o, dict):
        return {k: _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    return o


def save_metrics(name, notebook, **fields):
    rec = {"name": name, "notebook": notebook, "variant": VARIANT, "eval_variant": EVAL_VARIANT,
           "img": IMG, "smoke": SMOKE, "version": VERSION_STR, **fields}
    path = f"{models_folder}/{name}_metrics.json"
    with open(path, "w") as f:
        json.dump(_jsonable(rec), f, indent=2)
    print(f"-> {path}")
    return path

## Plots: confusion matrix and predicted vs. labelled

Used by both steps below, so Step 1 and Step 2 plots are defined identically.

**Detection confusion matrix.** A detector's errors aren't only "wrong class", so there is an
extra **background** row and column:
- predictions are kept at `score >= CONF_THR` (the same operating point as the P/R table);
- each ground-truth box is matched to at most one prediction with IoU ≥ 0.5, preferring a
  same-class prediction, then the highest IoU;
- a matched pair counts in `(true class, predicted class)`;
- an unmatched ground-truth box is a **miss**: `(true class, background)`;
- an unmatched prediction is a **false alarm**: `(background, predicted class)`. That includes
  a second, wrong-class head box on a head already matched.

The left panel shows counts. The right one divides each true-class row by its total, so its
diagonal is per-class recall at `CONF_THR`. For the drowsiness pipeline the cells that matter
are `closed_eye → open_eye` (a missed closure) and `head_down → head_straight` (a missed nod).

**Predicted vs. labelled.** Pairs of the same test image: ground truth on the left,
predictions at `CONF_THR` on the right. They are picked as a few images per class, plus the
images with the most errors (misses + false alarms + wrong classes), since those are the ones
worth looking at.

In [ ]:
def detection_confusion(preds, gts, conf=CONF_THR, iou_thr=0.5):
    """(NUM_CLASSES + 1)² counts; the last row/column is background (see the markdown)."""
    n = NUM_CLASSES
    cm = np.zeros((n + 1, n + 1), np.int64)
    for iid, (gb, gc) in gts.items():
        pb, ps, pc = preds.get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        keep = ps >= conf
        pb, pc = pb[keep], pc[keep]
        ious = iou_xyxy(gb, pb)
        g_used, p_used = np.zeros(len(gb), bool), np.zeros(len(pb), bool)
        pairs = [(gi, pi) for gi, pi in zip(*np.where(ious >= iou_thr))] if ious.size else []
        for gi, pi in sorted(pairs, key=lambda t: (gc[t[0]] != pc[t[1]], -ious[t])):
            if not g_used[gi] and not p_used[pi]:
                g_used[gi] = p_used[pi] = True
                cm[gc[gi], pc[pi]] += 1
        for gi in np.where(~g_used)[0]:
            cm[gc[gi], n] += 1
        for pi in np.where(~p_used)[0]:
            cm[n, pc[pi]] += 1
    return cm


def plot_confusion(cm, title):
    labels = CLASSES + ["background"]
    rows = cm.sum(axis=1, keepdims=True)
    norm = np.divide(cm, rows, out=np.zeros_like(cm, dtype=float), where=rows > 0)
    norm[-1] = 0                                  # background row has no "total" to normalise by
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
    for ax, data, fmt, sub in ((axes[0], cm, "d", "counts"), (axes[1], norm, ".2f", "row-normalised (recall)")):
        shown = data.astype(float).copy(); shown[-1, -1] = np.nan
        ax.imshow(shown, cmap="Blues")
        for i in range(len(labels)):
            for j in range(len(labels)):
                if i == j == len(labels) - 1 or (sub != "counts" and i == len(labels) - 1):
                    continue
                v = data[i, j]
                ax.text(j, i, format(v, fmt), ha="center", va="center", fontsize=8,
                        color="white" if (v / (data[:-1].max() or 1)) > 0.5 else "black")
        ax.set(xticks=range(len(labels)), yticks=range(len(labels)), xlabel="predicted", ylabel="true",
               title=f"{title}: {sub}")
        ax.set_xticklabels(labels, rotation=30, ha="right"); ax.set_yticklabels(labels)
    plt.tight_layout()
    return fig


def per_class_table(preds, df, cm):
    ev = evaluate(preds, gts_for(df))
    rows = {}
    for i, c in enumerate(CLASSES):
        p, r = ev["pr_at_conf"][c]["precision"], ev["pr_at_conf"][c]["recall"]
        rows[c] = {"GT boxes": int(cm[i].sum()), "AP50": ev["ap50"][c], "P": p, "R": r,
                   "F1": 2 * p * r / max(p + r, 1e-9),
                   "missed": int(cm[i, -1]), "false alarms": int(cm[-1, i]),
                   "wrong class": int(cm[i, :-1].sum() - cm[i, i])}
    return ev, pd.DataFrame(rows).T


def image_errors(preds, gts):
    """Per image: misses + false alarms + wrong-class matches at CONF_THR."""
    out = {}
    for iid, g in gts.items():
        cm = detection_confusion(preds, {iid: g})
        out[iid] = int(cm.sum() - np.trace(cm[:-1, :-1]))
    return out


def pick_examples(df, preds, n_per_class=2, n_worst=6):
    ids = []
    for i in range(NUM_CLASSES):
        cand = df[df.classes.apply(lambda a: bool((a == i).any())) & ~df.image_id.isin(ids)]
        ids += list(cand.sample(min(n_per_class, len(cand)), random_state=SEED).image_id)
    errs = image_errors(preds, gts_for(df))
    worst = [iid for iid, e in sorted(errs.items(), key=lambda t: -t[1]) if e > 0 and iid not in ids]
    return ids, worst[:n_worst], errs


def show_pred_vs_label(models_preds, df, ids, errs=None, title="", pairs_per_row=None):
    """One column of ground truth plus one per model in `models_preds` ({name: preds})."""
    by_id = df.set_index("image_id", drop=False)
    names = list(models_preds)
    per_img = 1 + len(names)
    per_row = pairs_per_row or max(1, 8 // per_img)
    n_rows = int(np.ceil(len(ids) / per_row))
    fig, axes = plt.subplots(n_rows, per_row * per_img, figsize=(2.1 * per_row * per_img, 2.4 * n_rows),
                             squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for k, iid in enumerate(ids):
        r = by_id.loc[iid]
        img = load_rgb(r)
        row, col = divmod(k, per_row)
        col *= per_img
        e = f" · {errs[iid]} err" if errs and iid in errs else ""
        draw_boxes(axes[row, col], img, gt_xyxy(r), r["classes"], title=f"LABEL{e}\n{r['project']}")
        for m, name in enumerate(names):
            pb, ps, pc = models_preds[name].get(iid, (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
            k_ = ps >= CONF_THR
            draw_boxes(axes[row, col + 1 + m], img, pb[k_], pc[k_], ps[k_], title=f"PRED {name}")
    fig.suptitle(title, y=1.0)
    plt.tight_layout()
    return fig


STEP_RESULTS = {}


def step_results(name, model, df=None):
    """Test-set predictions of a trained step → table, confusion matrix, image comparisons."""
    df = splits["test"] if df is None else df
    preds = ultra_predictions(model, df)
    cm = detection_confusion(preds, gts_for(df))
    ev, table = per_class_table(preds, df, cm)
    print(f"== {name} / TEST at CONF_THR={CONF_THR}: mAP50 {ev['map50']:.3f}, mAP50-95 {ev['map50_95']:.3f}, "
          f"eye state acc {ev['eye']['state_acc']:.3f} (recall {ev['eye']['recall']:.3f}), "
          f"head acc {ev['head']['state_acc']:.3f} (recall {ev['head']['recall']:.3f})")
    print(table.round(3).to_string())
    fig = plot_confusion(cm, f"{name} / test"); savefig(fig, f"{name}_confusion"); plt.show()

    ids, worst, errs = pick_examples(df, preds)
    fig = show_pred_vs_label({name: preds}, df, ids, errs, f"{name}: a few test images per class")
    savefig(fig, f"{name}_pred_vs_label"); plt.show()
    if worst:
        fig = show_pred_vs_label({name: preds}, df, worst, errs, f"{name}: test images with the most errors")
        savefig(fig, f"{name}_worst"); plt.show()
    STEP_RESULTS[name] = {"preds": preds, "cm": cm, "eval": ev, "table": table, "ids": ids, "worst": worst}
    return STEP_RESULTS[name]

## Dataset files for Ultralytics

Ultralytics accepts a `.txt` list of image paths per split and finds each label by swapping
`/images/` for `/labels/`. Writing the lists from `splits` means its training sees exactly the
images the Setup filters selected (close-ups, pseudo-eyes, smoke subset), the same as 12/14.

Training images follow `VARIANT`:
- `gray` or `ir` selects that folder;
- `mix` lists **both** versions of every training image. That's the closest file-based
  equivalent of 12/14's random per-image blend, and Ultralytics can't blend.

Validation and test use `EVAL_VARIANT`.

In [ ]:
def write_list(df, variant, name):
    path = f"{WORK_DIR}/yolo_{name}.txt"
    rows = [image_path(r, variant) for _, r in df.iterrows()]
    with open(path, "w") as f:
        f.write("\n".join(rows) + "\n")
    return path


train_variants = ["gray", "ir"] if VARIANT == "mix" else [VARIANT]
train_lists = [write_list(splits["train"], v, f"train_{v}") for v in train_variants]
YOLO_YAML = f"{WORK_DIR}/parts_det_ultralytics.yaml"
with open(YOLO_YAML, "w") as f:
    f.write(f"train: {json.dumps(train_lists)}\n"
            f"val: {write_list(splits['valid'], EVAL_VARIANT, 'valid')}\n"
            f"test: {write_list(splits['test'], EVAL_VARIANT, 'test')}\n"
            f"nc: {NUM_CLASSES}\nnames: {json.dumps(CLASSES)}\n")
print(open(YOLO_YAML).read())

## Training and evaluation

Fine-tuning settings:
- `imgsz=IMG` and default mosaic;
- `fliplr=0.5`, because the classes have no left/right;
- `hsv_s=hsv_h=0`, because the images are gray;
- early stopping with patience 20 on Ultralytics' own val fitness.

For scoring, predictions are taken at `conf=0.001` with NMS IoU 0.5, matching what 12/14 hand to
the shared evaluation. They are converted to the common `{image_id: (xyxy, scores, classes)}`
form, so mAP, eye-state and head numbers are defined exactly as in the other notebooks.
Ultralytics' own mAP is printed too, for reference only.

In [ ]:
from ultralytics import YOLO


def ultra_predictions(model, df, variant=None):
    preds = {}
    paths_ = [image_path(r, variant or EVAL_VARIANT) for _, r in df.iterrows()]
    for k in range(0, len(paths_), 64):
        for iid, res in zip(df.image_id[k:k + 64],
                            model.predict(paths_[k:k + 64], imgsz=IMG, conf=0.001, iou=0.5,
                                          max_det=30, verbose=False)):
            b = res.boxes
            preds[iid] = (b.xyxyn.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int))
    return preds


EXPORT_TIMEOUT_S = int(os.environ.get("ARGUS_EXPORT_TIMEOUT", "1800"))


def export_int8_tflite(pt_path):
    """Ultralytics' TFLite export pip-installs its converters on first use, and that install can
    hang (seen locally: >15 min stuck in `uv pip install litert-torch`). A hang never raises, so
    the export runs in a child process group with a time limit. On timeout or failure it returns
    None, and the notebook continues without int8 numbers for this model."""
    import signal
    code = ("import sys; from ultralytics import YOLO; "
            "print('EXPORTED=' + str(YOLO(sys.argv[1]).export(format='tflite', int8=True, "
            "imgsz=int(sys.argv[2]), data=sys.argv[3], verbose=False)))")
    proc = subprocess.Popen([sys.executable, "-c", code, pt_path, str(IMG), YOLO_YAML],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            start_new_session=True)
    try:
        log, _ = proc.communicate(timeout=EXPORT_TIMEOUT_S)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)   # also kills any pip install it started
        proc.communicate()
        print(f"int8 export timed out after {EXPORT_TIMEOUT_S}s; skipping int8 for {pt_path}")
        return None
    hits = [l.split("=", 1)[1].strip() for l in log.splitlines() if l.startswith("EXPORTED=")]
    if proc.returncode != 0 or not hits:
        print(f"int8 export failed (exit {proc.returncode}):\n" + "\n".join(log.splitlines()[-15:]))
        return None
    return hits[-1]


def run_ultralytics(weights, epochs=100):
    name = weights.replace(".pt", "")
    model = YOLO(weights)
    model.train(data=YOLO_YAML, imgsz=IMG, epochs=1 if SMOKE else epochs, batch=64,
                patience=20, fliplr=0.5, hsv_h=0.0, hsv_s=0.0, seed=SEED, deterministic=True,
                project=f"{WORK_DIR}/ultralytics", name=name, exist_ok=True,
                workers=2 if SMOKE else 8, plots=False, verbose=False)
    best = YOLO(f"{WORK_DIR}/ultralytics/{name}/weights/best.pt")
    val = report(ultra_predictions(best, splits["valid"]), splits["valid"], f"{name} / valid")
    test_float = report(ultra_predictions(best, splits["test"]), splits["test"], f"{name} / TEST float")

    pt_path = f"{models_folder}/{name}_{VERSION_STR}.pt"
    shutil.copy(f"{WORK_DIR}/ultralytics/{name}/weights/best.pt", pt_path)
    tflite_path, test_int8, latency = None, None, None
    out = export_int8_tflite(f"{WORK_DIR}/ultralytics/{name}/weights/best.pt")
    if out:
        try:
            tflite_path = f"{models_folder}/{name}_{VERSION_STR}_int8.tflite"
            shutil.copy(out, tflite_path)
            test_int8 = report(ultra_predictions(YOLO(out, task="detect"), splits["test"]),
                               splits["test"], f"{name} / TEST int8")
            latency = bench_tflite(tflite_path)
        except Exception as e:
            print(f"int8 eval failed for {name}: {e!r}")

    n_params = sum(p.numel() for p in best.model.parameters())
    save_metrics(name, "13_yolo_nano", params=int(n_params),
                 int8_kb=round(os.path.getsize(tflite_path) / 1024, 1) if tflite_path else None,
                 val_map50=val["all"]["map50"], test_float=test_float, test_int8=test_int8,
                 latency_ms=latency, pt_path=pt_path, tflite_path=tflite_path)
    return best, val

## Step 1: YOLO11n

The current Ultralytics nano (COCO-pretrained, about 2.6 M parameters), fine-tuned with the
settings above. `run_ultralytics` trains, scores valid / test with the shared evaluation, exports
int8 TFLite, times it, and writes `yolo11n_metrics.json`.

In [ ]:
yolo11, yolo11_val = run_ultralytics("yolo11n.pt")

### Step 1 results: predictions on the test set

Per-class AP50 / P / R / F1 at `CONF_THR`, the detection confusion matrix, and predicted vs.
labelled test images (a few per class, then the worst ones). See "Plots" above for exactly how
the matrix is counted.

In [ ]:
yolo11_res = step_results("yolo11n", yolo11)

## Step 2: YOLOv5nu

The YOLOv5-nano lineage with Ultralytics' anchor-free "u" head (about 2.6 M parameters), trained
and scored exactly like Step 1, so the two are directly comparable. It writes
`yolov5nu_metrics.json`.

In [ ]:
yolo5, yolo5_val = run_ultralytics("yolov5nu.pt")

### Step 2 results: predictions on the test set

The same table, confusion matrix and image comparisons as Step 1.

In [ ]:
yolo5_res = step_results("yolov5nu", yolo5)

## Step 1 vs. Step 2

Both models on the same test images. The table compares the headline numbers from the shared
evaluation, with recall per class read off each confusion matrix's diagonal. The image grid
puts the label next to both models' predictions for the images either step got most wrong, so
it shows where they disagree.

The int8 / latency side of the comparison is in each `<name>_metrics.json` and in notebook 14's
table.

In [ ]:
if len(STEP_RESULTS) < 2:
    print(f"Run both steps first (have: {list(STEP_RESULTS)})")
else:
    cmp = {}
    for name, r in STEP_RESULTS.items():
        ev, cm = r["eval"], r["cm"]
        rec = np.diag(cm[:-1, :-1]) / np.maximum(cm[:-1].sum(axis=1), 1)
        cmp[name] = {"mAP50": ev["map50"], "mAP50-95": ev["map50_95"],
                     **{f"AP50 {c}": ev["ap50"][c] for c in CLASSES},
                     **{f"recall@conf {c}": rec[i] for i, c in enumerate(CLASSES)},
                     "eye state acc": ev["eye"]["state_acc"], "head acc": ev["head"]["state_acc"],
                     "false alarms": int(cm[-1].sum()), "missed": int(cm[:-1, -1].sum())}
    cmp = pd.DataFrame(cmp)
    print(cmp.round(3).to_string())

    names = list(STEP_RESULTS)
    ids = list(dict.fromkeys(STEP_RESULTS[names[0]]["worst"][:4] + STEP_RESULTS[names[1]]["worst"][:4]))
    if ids:
        fig = show_pred_vs_label({n: STEP_RESULTS[n]["preds"] for n in names}, splits["test"], ids,
                                 title="label vs. Step 1 vs. Step 2 (each step's worst test images)",
                                 pairs_per_row=2)
        savefig(fig, "step1_vs_step2"); plt.show()

## Download the results (only when not saving to Drive)

With `USE_DRIVE = False`, the models and metrics live only on this session's disk and vanish
when it ends. This cell zips `models/parts_det/` and downloads it to your computer. With Drive
on, it does nothing, because the files are already in `MyDrive/Argus/models/parts_det/`.

**Shared verbatim by notebooks 12 / 13 / 14.**

In [ ]:
if IN_COLAB and not USE_DRIVE and not SMOKE:
    from google.colab import files
    archive = shutil.make_archive(f"{WORK_DIR}/parts_det_results_{VERSION_STR}", "zip", models_folder)
    files.download(archive)
else:
    print(f"results are in {models_folder}")